<a href="https://colab.research.google.com/github/Raisanhasan9/sms-spam-svm/blob/main/SMS_Spam_SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===================== SMS Spam SVM - ALL IN ONE CELL =====================
import re, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

# ---------- 1) Load data (Kaggle spam.csv upload) ----------
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(fname, encoding="latin-1")[['v1', 'v2']]
df.columns = ['label', 'message']
df = df.drop_duplicates()
df['label'] = df['label'].map({'ham': 0, 'spam': 1})
print("Data:", df.shape, "| spam:", df['label'].sum(), "| ham:", (df['label'] == 0).sum())

# ---------- 2) Smart normalizer ----------
def normalize_text(text):
    t = text.lower()
    t = re.sub(r'(http\S+|www\.\S+)', ' urltoken ', t)
    t = re.sub(r'\S+@\S+', ' emailtoken ', t)
    t = re.sub(r'[£$€]\s?\d[\d,.]*', ' moneytoken ', t)
    t = re.sub(r'\b\d{5,}\b', ' longnumtoken ', t)
    t = re.sub(r'\d+', ' numtoken ', t)
    t = re.sub(r'[^\w\s!?]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

# ---------- 3) Handcrafted features ----------
class TextStats(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        rows = []
        for s in X:
            n = len(s) + 1
            low = s.lower()
            rows.append([
                np.log1p(len(s)), np.log1p(len(s.split())),
                sum(c.isdigit() for c in s) / n,
                sum(c.isupper() for c in s) / n,
                s.count('!'),
                int(bool(re.search(r'http|www\.|\.com|\.co\.uk', low))),
                int(bool(re.search(r'[£$€]', s))),
                int(bool(re.search(r'\b\d{5,}\b', s))),
            ])
        return np.array(rows, dtype=float)

# ---------- 4) Pipeline ----------
def build_pipeline(C=1.0, word_ngram=(1, 2)):
    feats = FeatureUnion([
        ('word', TfidfVectorizer(preprocessor=normalize_text, ngram_range=word_ngram,
                                 sublinear_tf=True, min_df=2)),
        ('char', TfidfVectorizer(preprocessor=normalize_text, analyzer='char_wb',
                                 ngram_range=(2, 5), sublinear_tf=True, min_df=3)),
        ('stats', Pipeline([('s', TextStats()), ('sc', StandardScaler())])),
    ])
    return Pipeline([('feats', feats),
                     ('svm', LinearSVC(C=C, class_weight='balanced', max_iter=20000))])

# ---------- 5) Split + CV ----------
X, y = df['message'], df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ---------- 6) Tuning (F1) ----------
grid = GridSearchCV(build_pipeline(),
                    {'svm__C': [0.1, 0.3, 1, 3],
                     'feats__word__ngram_range': [(1, 2), (1, 3)]},
                    cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)
best = grid.best_estimator_
print("Best params:", grid.best_params_, "| CV F1:", round(grid.best_score_, 4))

# ---------- 7) Threshold tuning (train CV only, no leakage) ----------
oof = cross_val_predict(
    build_pipeline(C=grid.best_params_['svm__C'],
                   word_ngram=grid.best_params_['feats__word__ngram_range']),
    X_train, y_train, cv=cv, method='decision_function', n_jobs=-1)
ths = np.linspace(-1, 1, 81)
THRESHOLD = ths[int(np.argmax([f1_score(y_train, oof > t) for t in ths]))]
print("Best threshold:", round(THRESHOLD, 3))

# ---------- 8) Final evaluation ----------
pred = (best.decision_function(X_test) > THRESHOLD).astype(int)
print("\nAccuracy :", round(accuracy_score(y_test, pred), 4))
print("Precision:", round(precision_score(y_test, pred), 4))
print("Recall   :", round(recall_score(y_test, pred), 4))
print("F1       :", round(f1_score(y_test, pred), 4))
print(classification_report(y_test, pred, target_names=['ham', 'spam']))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=['ham', 'spam']).plot(cmap='Blues')
plt.show()

# ---------- 9) Error analysis ----------
err = pd.DataFrame({'message': X_test.values, 'true': y_test.values, 'pred': pred})
err = err[err.true != err.pred]
print("Spam ke ham bolse:", (err.true == 1).sum())
for m in err[err.true == 1].message.head(8): print("  -", m[:120])
print("Ham ke spam bolse:", (err.true == 0).sum())
for m in err[err.true == 0].message.head(8): print("  -", m[:120])

# ---------- 10) Predict function ----------
def predict_sms(text, threshold=THRESHOLD):
    score = best.decision_function([text])[0]
    return ("Spam" if score > threshold else "Ham"), round(float(score), 3)

print(predict_sms("Congratulations! You have won a free iPhone. Click here now."))
print(predict_sms("Hi, are we meeting tomorrow at 3 PM?"))
print(predict_sms("URGENT! Call 09061701461 to claim your £500 prize"))